# In Class Activity April 28th, 2026

### Irrigation Need with Gaussian Naive Bayes

In [1]:
# importing libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score,f1_score,balanced_accuracy_score,classification_report


### Importing, inspecting & splitting data

In [2]:
# load kaggle train data, inspect it, and split it
irrigation_data=pd.read_csv("/Users/christinewu/Downloads/playground-series-s6e4/train.csv")

print(irrigation_data.head())
print()
print(irrigation_data["Irrigation_Need"].value_counts())
print()
print(irrigation_data.isna().sum())

X_irrigation=irrigation_data.drop(["id","Irrigation_Need"],axis=1)
y_irrigation=irrigation_data["Irrigation_Need"]

X_train_irrigation,X_test_irrigation,y_train_irrigation,y_test_irrigation=train_test_split(
    X_irrigation,y_irrigation,test_size=0.20,stratify=y_irrigation,random_state=42
)


   id Soil_Type  Soil_pH  ...  Previous_Irrigation_mm  Region  Irrigation_Need
0   0     Loamy     4.92  ...                  112.16    East              Low
1   1      Clay     7.08  ...                   47.16   South              Low
2   2      Clay     5.69  ...                  110.38   North              Low
3   3     Sandy     5.65  ...                   53.85   South           Medium
4   4      Clay     7.96  ...                   93.19   South              Low

[5 rows x 21 columns]

Irrigation_Need
Low       369917
Medium    239074
High       21009
Name: count, dtype: int64

id                         0
Soil_Type                  0
Soil_pH                    0
Soil_Moisture              0
Organic_Carbon             0
Electrical_Conductivity    0
Temperature_C              0
Humidity                   0
Rainfall_mm                0
Sunlight_Hours             0
Wind_Speed_kmh             0
Crop_Type                  0
Crop_Growth_Stage          0
Season                     0
Ir

### Defining functions

In [3]:
# helper function for held out test evaluation
def evaluate_models_test(models,X_train,X_test,y_train,y_test):
    trained_models={}
    test_results=[]
    for name,model in models.items():
        model.fit(X_train,y_train)
        trained_models[name]=model
        preds=model.predict(X_test)
        test_results.append({
            "model":name,
            "test_accuracy":accuracy_score(y_test,preds),
            "test_f1_macro":f1_score(y_test,preds,average="macro"),
            "test_balanced_accuracy":balanced_accuracy_score(y_test,preds)
        })
    return trained_models,pd.DataFrame(test_results).sort_values("test_balanced_accuracy",ascending=False)

# helper function for multiclass threshold tuning on one class
def evaluate_class_thresholds(y_true,probs,classes,target_class,thresholds=None):
    if thresholds is None:
        thresholds=np.linspace(0.10,0.90,17)
    baseline_preds=np.array(classes)[probs.argmax(axis=1)]
    class_index=list(classes).index(target_class)
    rows=[]
    for t in thresholds:
        preds=baseline_preds.copy()
        preds[probs[:,class_index]>=t]=target_class
        report=classification_report(y_true,preds,output_dict=True,zero_division=0)
        rows.append({
            "threshold":t,
            "precision":report[target_class]["precision"],
            "recall":report[target_class]["recall"],
            "f1":report[target_class]["f1-score"],
            "balanced_accuracy":balanced_accuracy_score(y_true,preds)
        })
    return pd.DataFrame(rows)


### Identifying categorical/numerical & defining model

In [6]:
# identify numeric and categorical columns and define irrigation models
numeric_features_irrigation=X_train_irrigation.select_dtypes(include=["int64","float64"]).columns.tolist()
categorical_features_irrigation=X_train_irrigation.select_dtypes(include=["object"]).columns.tolist()

print("numeric columns:",numeric_features_irrigation)
print("categorical columns:",categorical_features_irrigation)

numeric_transformer_nb=SimpleImputer(strategy="median")

numeric_transformer_lr=Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler())
])

categorical_transformer=Pipeline([
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("ordinal",OrdinalEncoder(handle_unknown="use_encoded_value",unknown_value=-1))
])

preprocessor_nb=ColumnTransformer([
    ("num",numeric_transformer_nb,numeric_features_irrigation),
    ("cat",categorical_transformer,categorical_features_irrigation)
],sparse_threshold=0)

preprocessor_lr=ColumnTransformer([
    ("num",numeric_transformer_lr,numeric_features_irrigation),
    ("cat",categorical_transformer,categorical_features_irrigation)
],sparse_threshold=0)

models_irrigation={
    "Naive Bayes":Pipeline([
        ("preprocessor",preprocessor_nb),
        ("model",GaussianNB())
    ]),
    "Logistic Regression":Pipeline([
        ("preprocessor",preprocessor_lr),
        ("model",LogisticRegression(max_iter=500))
    ])
}


numeric columns: ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']
categorical columns: ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']


### Building the Gaussian Naive Bayes model on the Kaggle train data

In [7]:
# fit irrigation models and compare them on the held out test split
trained_models_irrigation,test_results_irrigation=evaluate_models_test(
    models_irrigation,X_train_irrigation,X_test_irrigation,y_train_irrigation,y_test_irrigation
)

test_results_irrigation


,model,test_accuracy,test_f1_macro,test_balanced_accuracy
0,Naive Bayes,0.827516,0.725962,0.691060
1,Logistic Regression,0.748833,0.647709,0.613465


### Generating predicted probabilities

In [8]:
# generate predicted probabilities with naive bayes
model_nb=trained_models_irrigation["Naive Bayes"]
probs_nb=model_nb.predict_proba(X_test_irrigation)
classes_nb=model_nb.named_steps["model"].classes_

probs_nb_df=pd.DataFrame(probs_nb,columns=classes_nb)
probs_nb_df.head()


,High,Low,Medium
0,2.647267e-13,0.935411,0.064589
1,6.926318e-04,0.297763,0.701544
2,2.102430e-07,0.539037,0.460962
3,3.689770e-07,0.884064,0.115935
4,9.592931e-09,0.749994,0.250006


### Creating baseline predictions with the default rule

In [9]:
# baseline predictions use the class with the highest probability
preds_baseline=np.array(classes_nb)[probs_nb.argmax(axis=1)]

print("baseline balanced accuracy:",round(balanced_accuracy_score(y_test_irrigation,preds_baseline),4))
print()
print(classification_report(y_test_irrigation,preds_baseline,zero_division=0))


baseline balanced accuracy: 0.6911

              precision    recall  f1-score   support

        High       0.72      0.42      0.53      4202
         Low       0.85      0.90      0.88     73983
      Medium       0.79      0.75      0.77     47815

    accuracy                           0.83    126000
   macro avg       0.79      0.69      0.73    126000
weighted avg       0.82      0.83      0.82    126000



### Choosing one class and one metric

In [11]:
# focus on the rare high irrigation class and use f1 as the metric
target_class="High"
metric_name="F1"

print("selected class:",target_class)
print("selected metric:",metric_name)
print()
print(y_irrigation.value_counts(normalize=True).round(4))


selected class: High
selected metric: F1

Irrigation_Need
Low       0.5872
Medium    0.3795
High      0.0333
Name: proportion, dtype: float64


### Applying a threshold to the chosen class

In [12]:
# test thresholds for the high irrigation class
threshold_results_high=evaluate_class_thresholds(
    y_test_irrigation,probs_nb,classes_nb,target_class
)

threshold_results_high.sort_values("f1",ascending=False).head()


,threshold,precision,recall,f1,balanced_accuracy
3,0.25,0.548499,0.752261,0.634420,0.788033
4,0.30,0.587769,0.688482,0.634152,0.770503
5,0.35,0.629317,0.620181,0.624715,0.751026
2,0.20,0.488019,0.794860,0.604744,0.796600
6,0.40,0.666861,0.544979,0.599790,0.728559


### Generating a second set of predictions with the threshold

In [13]:
# apply the chosen threshold to the high irrigation class
threshold_high=0.25
high_index=list(classes_nb).index(target_class)

preds_threshold=preds_baseline.copy()
preds_threshold[probs_nb[:,high_index]>=threshold_high]=target_class

print("baseline high-class f1:",round(f1_score(y_test_irrigation,preds_baseline,labels=[target_class],average="macro"),4))
print("threshold high-class f1:",round(f1_score(y_test_irrigation,preds_threshold,labels=[target_class],average="macro"),4))
print()
print("baseline balanced accuracy:",round(balanced_accuracy_score(y_test_irrigation,preds_baseline),4))
print("threshold balanced accuracy:",round(balanced_accuracy_score(y_test_irrigation,preds_threshold),4))
print()
print("Baseline classification report")
print(classification_report(y_test_irrigation,preds_baseline,zero_division=0))
print("Threshold classification report")
print(classification_report(y_test_irrigation,preds_threshold,zero_division=0))


baseline high-class f1: 0.5332
threshold high-class f1: 0.6344

baseline balanced accuracy: 0.6911
threshold balanced accuracy: 0.788

Baseline classification report
              precision    recall  f1-score   support

        High       0.72      0.42      0.53      4202
         Low       0.85      0.90      0.88     73983
      Medium       0.79      0.75      0.77     47815

    accuracy                           0.83    126000
   macro avg       0.79      0.69      0.73    126000
weighted avg       0.82      0.83      0.82    126000

Threshold classification report
              precision    recall  f1-score   support

        High       0.55      0.75      0.63      4202
         Low       0.85      0.90      0.88     73983
      Medium       0.81      0.71      0.75     47815

    accuracy                           0.82    126000
   macro avg       0.74      0.79      0.76    126000
weighted avg       0.83      0.82      0.82    126000



### Discussion
I focused on the High irrigation class because it is the rarest class in the training data. I used F1 for this class because it balances precision and recall.

I chose a threshold of 0.25 for the High class. With the default rule, the High class F1 was about 0.53. After applying the threshold, the High class F1 increased to about 0.63. The recall for High improved a lot, while precision decreased. The overall balanced accuracy also increased from about 0.69 to about 0.79.

The tradeoff is that the threshold makes the model assign High more often. That helps catch more truly high-irrigation cases, but it also creates more false positives for that class.

Compared with the logistic regression model in this notebook, Naive Bayes performed better on balanced accuracy for this split. That means Naive Bayes handled the class imbalance better here, especially after thresholding the High class.
